In [1]:
import os
import numpy as np
import pandas as pd
import joblib
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, recall_score, f1_score
import torch

path = '../../dataset/preprocessed/hotel_bookings_dummy.csv'
df = pd.read_csv(path)

X = df.drop(['is_canceled'], axis=1)
y = df['is_canceled']
X = pd.get_dummies(X, drop_first=True, dtype=float)

# 개별 모델들과 똑같은 기준으로 테스트 데이터를 분리합니다.
X_train, X_test, y_train, y_test = train_test_split(
    X, y, 
    train_size=0.7, 
    random_state=1004
)

print(f"📦 데이터 로드 완료! Test 데이터 크기: {X_test.shape}")


target_path = '../../visualization/model_results'

rf_model = joblib.load(f'{target_path}/best_rf.pkl')
xgb_model = joblib.load(f'{target_path}/best_xgb.pkl')
mlp_model = joblib.load(f'{target_path}/best_mlp.pkl')

print("✅ 모든 진짜 커스텀 모델(RF, XGB, MLP) 로드 성공!")

# ==========================================
# 3. 🔮 불러온 진짜 모델들로 가중치 보팅 앙상블 진행 (순수 CPU 모드)
# ==========================================
print("\n🔮 각 모델의 실전 데이터 예측 확률 추출 중...")

# (1) 머신러닝 계열 모델 확률 추출
rf_prob = rf_model.predict_proba(X_test)[:, 1]
xgb_prob = xgb_model.predict_proba(X_test)[:, 1]

# (2) 🐌 [메모리 버그 해결] MLP 모델을 확실하게 CPU 영역으로 고정
mlp_model.to('cpu')

# 입력 데이터 텐서도 가벼운 기본 CPU 텐서로 생성
X_test_tensor = torch.tensor(X_test.values, dtype=torch.float32)

# 모델을 평가 모드(eval)로 변경
mlp_model.eval()

with torch.no_grad(): # 그라디언트 계산 비활성화로 CPU 메모리 절약
    # 모델에 데이터를 통과시킨 후, 시그모이드 함수를 거쳐 0~1 사이의 취소 확률을 생성합니다.
    mlp_output = mlp_model(X_test_tensor)
    
    # CPU 연산 결과이므로 .cpu() 없이 바로 넘파이 변환 가능
    mlp_prob = torch.sigmoid(mlp_output).numpy().flatten()

print("✅ 파이토치 MLP를 포함한 모든 모델의 확률 추출 완료!")

# 🌟 [황금 가중치 결합] 에이스 모델인 XGBoost에 50%, RF에 30%, MLP에 20% 부여
voting_prob = (xgb_prob * 0.5) + (rf_prob * 0.3) + (mlp_prob * 0.2)
voting_preds = (voting_prob >= 0.5).astype(int)

# ==========================================
# 4. 📊 개별 모델 vs 가중치 보팅 앙상블 성적 전수조사
# ==========================================
print("\n🔍 각 개별 모델 및 앙상블의 최종 실전 성적 계산 중...")

rf_preds = (rf_prob >= 0.5).astype(int)
xgb_preds = (xgb_prob >= 0.5).astype(int)
mlp_preds = (mlp_prob >= 0.5).astype(int)

metrics_summary = []


for name, preds in zip(['Random Forest', 'XGBoost', 'MLP (Deep Learning)'], [rf_preds, xgb_preds, mlp_preds]):
    metrics_summary.append({
        'Model_Name': name,
        'Accuracy': accuracy_score(y_test, preds),
        'Recall': recall_score(y_test, preds),
        'F1_Score': f1_score(y_test, preds)
    })

# 최종 가중치 보팅 앙상블 점수 추가
metrics_summary.append({
    'Model_Name': '⭐ Weighted Voting Ensemble',
    'Accuracy': accuracy_score(y_test, voting_preds),
    'Recall': recall_score(y_test, voting_preds),
    'F1_Score': f1_score(y_test, voting_preds)
})

df_res = pd.DataFrame(metrics_summary)

print("\n" + "="*80)
print("모델 3개 vs 가중치 보팅 앙상블")
print("="*80)
display(df_res.round(4))  # 노트북 화면에 이쁜 표로 출력
print("="*80)

# 결과서 및 시각화를 위해 원래 저장하려던 visualization/model_results 폴더 구조로 맵핑
output_dir = '../../visualization/model_results'
os.makedirs(output_dir, exist_ok=True)
df_res.to_csv(f'{output_dir}/total_ensemble_result.csv', index=False)

# print(f"\n✅ 최종 비교 성적표가 '{output_dir}/total_ensemble_result.csv'로 저장되었습니다!")

: 